# Comparación de las corridas de Sextans A — filtro V

Este cuaderno compara el resumen anterior (`Resumen_SextansA_Filtro_V_Chips_1_a_8.csv`) con la corrida nueva guardada en `dataSextansA_multiplicadores_FWHM`. Las comparaciones se hacen por chip para los chips **2–8**, que están presentes en ambas corridas. El chip 1 queda fuera porque no tiene una corrida nueva.

Ejecuta las celdas en orden o selecciona **Run All**. El cuaderno crea las tablas y gráficas aquí mismo y guarda una copia de la tabla comparativa y las figuras en `dataSextansA_multiplicadores_FWHM/figuras_comparacion`.

> La corrida anterior del chip 2 fue manual (6 estrellas y 22 cajas de cielo); la nueva es automática (15 y 15). Además, para el chip 6 el anillo de cielo nuevo empieza a 7 × FWHM para no solaparse con la apertura de 6 × FWHM. Por eso las diferencias describen el cambio entre corridas completas; no siempre aíslan el efecto del radio de apertura por sí solo.

## 1. Preparar entorno y rutas

Se usan el CSV consolidado anterior y los productos detallados de la corrida nueva. Las métricas de la nueva corrida se reconstruyen desde los catálogos y resúmenes guardados por chip.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASELINE_FILE = "Resumen_SextansA_Filtro_V_Chips_1_a_8.csv"
PROJECT_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / BASELINE_FILE).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No encontré {BASELINE_FILE} desde {Path.cwd()}")

NEW_ROOT = PROJECT_ROOT / "dataSextansA_multiplicadores_FWHM"
NEW_CHIPS = NEW_ROOT / "chips"
FIG_DIR = NEW_ROOT / "figuras_comparacion"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CHIPS = list(range(2, 9))

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "font.size": 9,
})

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Corrida anterior: {PROJECT_ROOT / BASELINE_FILE}")
print(f"Corrida nueva: {NEW_ROOT}")

## 2. Reconstruir las métricas de la corrida nueva

Los conteos de DAOFIND, apertura válida, anillo válido, fotometría, selección de calidad y anillos adaptativos se cuentan desde los archivos por chip. Las métricas de diferencia fotométrica se leen de los resúmenes finales generados por `sextans_auto.py`.

In [2]:
PHOT_COLUMNS = ["x", "y", "msky", "stdev", "nsky", "flux", "mag", "merr"]
DAO_COLUMNS = ["x", "y", "mag_daofind", "sharpness", "sround", "ground", "id"]


def count_data_lines(path):
    """Cuenta filas de datos, ignorando líneas vacías y comentarios IRAF."""
    with Path(path).open(encoding="utf-8", errors="replace") as handle:
        return sum(
            1 for line in handle
            if line.strip() and not line.lstrip().startswith("#")
        )


def read_table(path):
    return pd.read_csv(path, sep=r"\s+", engine="python")


def load_new_chip(chip):
    folder = NEW_CHIPS / f"chip{chip}"
    prefix = f"V_chip{chip}"
    config = json.loads((folder / f"{prefix}_auto_config.json").read_text())
    final = json.loads((folder / f"{prefix}_resumen_final.json").read_text())
    threshold = json.loads((folder / f"{prefix}_threshold.json").read_text())

    catalog = pd.read_csv(
        folder / f"{prefix}_final.coo", comment="#", sep=r"\s+",
        names=DAO_COLUMNS, engine="python",
    )
    geometry = read_table(folder / f"{prefix}_geometria.dat")
    ap_valid = (
        geometry["dist_invalidos"].gt(config["r_ap"])
        & geometry["dist_borde"].gt(config["r_ap"])
    )

    phot = pd.read_csv(
        folder / f"{prefix}_fixed.dat", sep=r"\s+", names=PHOT_COLUMNS,
        na_values=["INDEF"], engine="python",
    )
    measured = phot["mag"].notna() & phot["merr"].notna()
    quality = measured & phot["merr"].le(0.20)

    isolated = read_table(folder / f"{prefix}_aisladas.dat")
    fixed_rings = read_table(folder / f"{prefix}_anillo_fijo.dat")
    f_out = fixed_rings["f_out"].to_numpy(dtype=float)
    q1, q3 = np.percentile(f_out, [25, 75])
    f_lim = float(q3 + 1.5 * (q3 - q1))
    clean_rings = fixed_rings["f_out"].le(f_lim)

    adaptive = read_table(folder / f"{prefix}_anillo_adaptativo.dat")
    accepted = adaptive["estado"].eq("aceptado")
    adaptive_r_in = pd.to_numeric(adaptive["r_in"], errors="coerce")
    moved_ring = accepted & adaptive_r_in.gt(config["r_sky_in"] + 1e-8)
    comparison = pd.read_csv(folder / f"{prefix}_comparacion.csv")
    control = comparison[np.isclose(comparison["r_in"], config["r_sky_in"])]
    delta_columns = []
    for name in ["msky", "stdev", "mag", "merr", "flux"]:
        delta = (control[f"{name}_adapt"] - control[f"{name}_fixed"]).abs().dropna()
        if len(delta):
            delta_columns.append(float(delta.max()))
    control_delta_max = max(delta_columns, default=np.nan)

    return {
        "chip": chip,
        "fwhm": config["fwhm"],
        "sigma_sky": config["sigma_sky"],
        "threshold": threshold["threshold"],
        "r2": threshold["r2"],
        "multiplo_ap": config["ap_fwhm_multiplier"],
        "radio_apertura": config["r_ap"],
        "radio_cielo_int": config["r_sky_in"],
        "radio_cielo_ext": config["r_sky_out"],
        "dao": len(catalog),
        "apertura": int(ap_valid.sum()),
        "anillo_valido": count_data_lines(folder / f"{prefix}_phot_ready.xy"),
        "phot_total": len(phot),
        "phot_validas": int(measured.sum()),
        "merr": int(quality.sum()),
        "estrellas_muestra": config["n_estrellas"],
        "cajas_cielo": config["n_cajas"],
        "aisladas": len(isolated),
        "analizadas": len(fixed_rings),
        "f_lim": f_lim,
        "limpias": int(clean_rings.sum()),
        "contaminadas": int((~clean_rings).sum()),
        "adapt_aceptadas": int(accepted.sum()),
        "recuperadas": int(moved_ring.sum()),
        "sin_solucion": int((~accepted).sum()),
        "control": final["N_control"],
        "movidas_validas": final["N_movidas_validas"],
        "match_max": float(comparison["dist_match"].max()),
        "control_delta_max": control_delta_max,
        "delta_mag": final["med_delta_mag"],
        "abs_delta_mag": final["med_abs_delta_mag"],
        "merr_fijo": final["med_merr_fixed"],
        "merr_adapt": final["med_merr_adapt"],
        "pct_mejora_merr": final["pct_merr_mejora"],
        "snr_fijo": final["med_snr_fixed"],
        "snr_adapt": final["med_snr_adapt"],
        "ratio_flujo": final["med_flux_ratio"],
        "fout_inicial": final["med_fout_inicial"],
        "fout_final": final["med_fout_final"],
        "delta_fout": final["med_delta_fout"],
        "pct_fout_reduce": final["pct_fout_reduce"],
    }


new = pd.DataFrame([load_new_chip(chip) for chip in CHIPS]).sort_values("chip")
new.head()

## 3. Unir las corridas y revisar la tabla

Las diferencias `*_nuevo_menos_anterior` se expresan en las unidades de cada variable: píxeles, conteos, magnitudes, errores o fracciones. En las gráficas de contaminación, las fracciones se convierten a porcentaje o puntos porcentuales.

In [3]:
baseline = pd.read_csv(PROJECT_ROOT / BASELINE_FILE, encoding="utf-8-sig")
baseline = baseline[baseline["chip"].isin(CHIPS)].copy()

metrics = [
    "fwhm", "sigma_sky", "threshold", "r2", "multiplo_ap",
    "radio_apertura", "radio_cielo_int", "radio_cielo_ext",
    "dao", "apertura", "anillo_valido", "phot_total", "phot_validas", "merr",
    "estrellas_muestra", "cajas_cielo", "aisladas", "analizadas", "f_lim",
    "limpias", "contaminadas", "adapt_aceptadas", "recuperadas", "sin_solucion",
    "control", "movidas_validas", "match_max", "control_delta_max",
    "delta_mag", "abs_delta_mag", "merr_fijo", "merr_adapt", "pct_mejora_merr",
    "snr_fijo", "snr_adapt", "ratio_flujo", "fout_inicial", "fout_final",
    "delta_fout", "pct_fout_reduce",
]
old_column = {
    "fwhm": "fwhm", "sigma_sky": "sigma_sky", "threshold": "threshold", "r2": "r2",
    "radio_apertura": "radio_apertura", "radio_cielo_int": "radio_cielo_int",
    "radio_cielo_ext": "radio_cielo_ext", "dao": "dao", "apertura": "apertura",
    "anillo_valido": "anillo_valido", "phot_total": "phot_total",
    "phot_validas": "phot_validas", "merr": "merr", "estrellas_muestra": "estrellas_muestra",
    "cajas_cielo": "cajas_cielo", "aisladas": "aisladas", "analizadas": "analizadas",
    "limpias": "limpias", "contaminadas": "contaminadas",
    "adapt_aceptadas": "adapt_aceptadas", "recuperadas": "recuperadas",
    "sin_solucion": "sin_solucion", "control": "control", "movidas_validas": "movidas_validas",
    "match_max": "match_max", "control_delta_max": "control_delta_max",
    "delta_mag": "delta_mag", "abs_delta_mag": "abs_delta_mag", "merr_fijo": "merr_fijo",
    "merr_adapt": "merr_adapt", "pct_mejora_merr": "pct_mejora_merr",
    "snr_fijo": "snr_fijo", "snr_adapt": "snr_adapt", "ratio_flujo": "ratio_flujo",
    "fout_inicial": "fout_inicial", "fout_final": "fout_final", "delta_fout": "delta_fout",
    "pct_fout_reduce": "pct_fout_reduce", "f_lim": "f_lim",
}
old_view = pd.DataFrame({"chip": baseline["chip"], "modo_anterior": baseline["modo"]})
for metric in metrics:
    source = old_column.get(metric, metric)
    if source in baseline.columns:
        old_view[f"{metric}_anterior"] = baseline[source].to_numpy()
    else:
        old_view[f"{metric}_anterior"] = np.nan
old_view["multiplo_ap_anterior"] = baseline["radio_apertura"].to_numpy() / baseline["fwhm"].to_numpy()
new_view = new[["chip", *metrics]].rename(columns={m: f"{m}_nuevo" for m in metrics})
new_view["modo_nuevo"] = "Automático (15 estrellas, 15 cajas)"
comparison = old_view.merge(new_view, on="chip", validate="one_to_one").sort_values("chip")

for metric in metrics:
    old_name, new_name = f"{metric}_anterior", f"{metric}_nuevo"
    if old_name in comparison and new_name in comparison:
        comparison[f"{metric}_nuevo_menos_anterior"] = comparison[new_name] - comparison[old_name]

comparison.to_csv(NEW_ROOT / "comparacion_corridas_chip2_8.csv", index=False)

visible = [
    "chip", "modo_anterior", "fwhm_anterior", "fwhm_nuevo",
    "multiplo_ap_nuevo", "radio_apertura_anterior", "radio_apertura_nuevo",
    "radio_cielo_int_anterior", "radio_cielo_int_nuevo",
    "dao_anterior", "dao_nuevo", "control_anterior", "control_nuevo",
    "delta_mag_anterior", "delta_mag_nuevo", "merr_adapt_anterior", "merr_adapt_nuevo",
]
display(comparison[visible].round(4))

Chip,Modo anterior,FWHM ant.,FWHM nueva,Apertura × FWHM,r_ap ant. (pix),r_ap nueva (pix),r_cielo ant. (pix),r_cielo nuevo (pix),DAO ant.,DAO nuevo,Control ant.,Control nuevo,Δmag ant.,Δmag nueva,MERR adapt. ant.,MERR adapt. nueva
2,Manual IRAF,4.52,4.48,2.5,13.56,11.200,22.60,22.40,7291,7645,321,352,-0.0100,-0.0070,0.0320,0.0390
3,Automático,4.62,4.62,1.5,13.86,6.930,23.10,23.10,2561,2561,353,402,-0.0040,-0.0060,0.0400,0.0300
4,Automático,4.61,4.61,1.5,13.83,6.915,23.05,23.05,1856,1856,283,330,-0.0020,-0.0010,0.0330,0.0225
5,Automático,4.48,4.48,2.0,13.44,8.960,22.40,22.40,6150,6150,789,1471,-0.0075,0.0190,0.0470,0.0585
6,Automático,4.62,4.62,6.0,13.86,27.720,23.10,32.34,1570,1570,302,199,-0.0060,0.0000,0.0290,0.0305
7,Automático,4.42,4.42,1.0,13.26,4.420,22.10,22.10,9719,9719,282,369,-0.0085,-0.0025,0.0505,0.0310
8,Automático,4.48,4.48,1.5,13.44,6.720,22.40,22.40,17735,17735,384,563,-0.0130,-0.0050,0.0400,0.0470


## 4. Geometría fotométrica y detección

La corrida anterior usó la apertura indicada en el resumen original; la nueva emplea los múltiplos solicitados. También se comparan FWHM, anillos de cielo y el ajuste de threshold DAOFIND.

In [4]:
def paired_slope(ax, metric, title, ylabel, scale=1.0, reference=None):
    x = np.arange(len(comparison))
    old_values = comparison[f"{metric}_anterior"].to_numpy(dtype=float) * scale
    new_values = comparison[f"{metric}_nuevo"].to_numpy(dtype=float) * scale
    for i, (old_value, new_value) in enumerate(zip(old_values, new_values)):
        ax.plot([i - 0.12, i + 0.12], [old_value, new_value], color="#B8B8B8", lw=1, zorder=1)
    ax.scatter(x - 0.12, old_values, color="#3B6FB6", marker="o", s=34, label="Anterior", zorder=2)
    ax.scatter(x + 0.12, new_values, color="#E07A27", marker="s", s=34, label="Nueva", zorder=2)
    ax.set_xticks(x, comparison["chip"].astype(str))
    ax.set_xlabel("Chip")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    if reference is not None:
        ax.axhline(reference, color="#555555", ls="--", lw=1)
    ax.legend(frameon=False)


def paired_bars(ax, metric, title, ylabel):
    chips = comparison["chip"].astype(str).to_list()
    x = np.arange(len(chips))
    width = 0.36
    old_values = comparison[f"{metric}_anterior"].to_numpy(dtype=float)
    new_values = comparison[f"{metric}_nuevo"].to_numpy(dtype=float)
    ax.bar(x - width / 2, old_values, width, color="#3B6FB6", label="Anterior")
    ax.bar(x + width / 2, new_values, width, color="#E07A27", label="Nueva")
    ax.set_xticks(x, chips)
    ax.set_xlabel("Chip")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)


def save_show(fig, filename):
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=170, bbox_inches="tight")
    plt.close(fig)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
settings = [
    ("fwhm", "FWHM mediano", "píxeles", 1, None),
    ("sigma_sky", "Ruido del cielo", "ADU", 1, None),
    ("multiplo_ap", "Múltiplo de apertura", "× FWHM", 1, None),
    ("radio_apertura", "Radio de apertura", "píxeles", 1, None),
    ("radio_cielo_int", "Radio interno del cielo", "píxeles", 1, None),
    ("radio_cielo_ext", "Radio externo del cielo", "píxeles", 1, None),
    ("threshold", "Threshold DAOFIND", "σ", 1, None),
    ("r2", "R² del ajuste de threshold", "R²", 1, None),
]
for ax, (metric, title, ylabel, scale, reference) in zip(axes.flat, settings):
    paired_slope(ax, metric, title, ylabel, scale, reference)
fig.suptitle("Configuración y detección: anterior frente a nueva", y=1.02, fontsize=14)
save_show(fig, "01_configuracion_y_deteccion.png")

![Configuración, geometría y threshold DAOFIND](dataSextansA_multiplicadores_FWHM/figuras_comparacion/01_configuracion_y_deteccion.png)

## 5. Conteos a lo largo del proceso

Los primeros paneles comparan detecciones y cortes de fotometría. Los siguientes muestran selección de fuentes, anillos de cielo limpios y resultados adaptativos. Los conteos mantienen la definición de cada salida guardada en la corrida.

In [5]:
fig, axes = plt.subplots(3, 4, figsize=(17, 12))
count_panels = [
    ("dao", "Detecciones DAOFIND", "fuentes"),
    ("apertura", "Apertura geométricamente válida", "fuentes"),
    ("anillo_valido", "Con anillo de cielo válido", "fuentes"),
    ("phot_total", "Mediciones PHOT", "fuentes"),
    ("phot_validas", "Magnitud y error definidos", "fuentes"),
    ("merr", "Pasan MERR ≤ 0.20", "fuentes"),
    ("aisladas", "Fuentes aisladas", "fuentes"),
    ("analizadas", "Anillos fijos medidos", "fuentes"),
    ("limpias", "Anillos fijos limpios", "fuentes"),
    ("contaminadas", "Anillos fijos contaminados", "fuentes"),
    ("adapt_aceptadas", "Aceptadas con anillo adaptativo", "fuentes"),
    ("sin_solucion", "Sin solución adaptativa", "fuentes"),
]
for ax, (metric, title, ylabel) in zip(axes.flat, count_panels):
    paired_bars(ax, metric, title, ylabel)
fig.suptitle("Conteos de detección, fotometría y selección", y=1.01, fontsize=14)
save_show(fig, "02_conteos_por_etapa.png")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
paired_bars(axes[0], "control", "Estrellas del grupo de control", "estrellas")
paired_bars(axes[1], "movidas_validas", "Fuentes con anillo desplazado y fotometría válida", "estrellas")
fig.suptitle("Muestra usada en la comparación fijo–adaptativo", y=1.03, fontsize=13)
save_show(fig, "03_muestra_de_comparacion.png")

![Conteos de detección, fotometría y selección](dataSextansA_multiplicadores_FWHM/figuras_comparacion/02_conteos_por_etapa.png)

![Tamaños de las muestras comparadas](dataSextansA_multiplicadores_FWHM/figuras_comparacion/03_muestra_de_comparacion.png)

## 6. Diferencias fotométricas y calidad

`Δ magnitud` es mediana de (magnitud adaptativa − magnitud fija) para las fuentes con anillo desplazado. `|Δ magnitud|` es la mediana del valor absoluto de esa diferencia. Los paneles de `MERR` y SNR comparan las medianas fija/adaptativa reportadas en cada corrida.

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
paired_slope(axes[0, 0], "delta_mag", "Δ magnitud mediana", "mag", reference=0)
paired_slope(axes[0, 1], "abs_delta_mag", "|Δ magnitud| mediana", "mag")
paired_slope(axes[0, 2], "merr_fijo", "MERR mediano — fijo", "mag")
paired_slope(axes[1, 0], "merr_adapt", "MERR mediano — adaptativo", "mag")
paired_slope(axes[1, 1], "pct_mejora_merr", "Fuentes con MERR mejorado", "%")
paired_slope(axes[1, 2], "ratio_flujo", "Razón de flujo adaptativo/fijo", "razón", reference=1)
fig.suptitle("Resultados fotométricos sobre fuentes con anillo desplazado", y=1.02, fontsize=14)
save_show(fig, "04_resultados_fotometricos.png")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
paired_slope(axes[0], "snr_fijo", "SNR mediano — fijo", "SNR")
paired_slope(axes[1], "snr_adapt", "SNR mediano — adaptativo", "SNR")
fig.suptitle("Relación señal/ruido", y=1.03, fontsize=13)
save_show(fig, "05_snr.png")

![Resultados fotométricos](dataSextansA_multiplicadores_FWHM/figuras_comparacion/04_resultados_fotometricos.png)

![Relación señal/ruido](dataSextansA_multiplicadores_FWHM/figuras_comparacion/05_snr.png)

## 7. Contaminación del anillo de cielo

`f_out` se expresa como porcentaje. `Δ f_out` se muestra en puntos porcentuales; valores negativos indican que el anillo final adaptativo tuvo menos contaminación que el anillo inicial para las fuentes comparadas.

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
paired_slope(axes[0], "fout_inicial", "Contaminación inicial mediana", "%", scale=100)
paired_slope(axes[1], "fout_final", "Contaminación final mediana", "%", scale=100)
paired_slope(axes[2], "delta_fout", "Cambio mediano de contaminación", "puntos porcentuales", scale=100, reference=0)
fig.suptitle("Contaminación de los anillos de cielo", y=1.03, fontsize=13)
save_show(fig, "06_contaminacion_cielo.png")

![Contaminación de los anillos de cielo](dataSextansA_multiplicadores_FWHM/figuras_comparacion/06_contaminacion_cielo.png)

## 8. Eficiencia relativa de las etapas

Las proporciones ayudan a separar cambios en el tamaño de la muestra de cambios en las métricas fotométricas. Se calculan como porcentajes a partir de los conteos de cada corrida.

In [8]:
for run in ["anterior", "nuevo"]:
    comparison[f"tasa_phot_validas_{run}"] = (
        100 * comparison[f"phot_validas_{run}"] / comparison[f"phot_total_{run}"].replace(0, np.nan)
    )
    comparison[f"tasa_merr_{run}"] = (
        100 * comparison[f"merr_{run}"] / comparison[f"phot_validas_{run}"].replace(0, np.nan)
    )
    comparison[f"tasa_cielo_limpio_{run}"] = (
        100 * comparison[f"limpias_{run}"] / comparison[f"analizadas_{run}"].replace(0, np.nan)
    )
    comparison[f"tasa_adapt_aceptada_{run}"] = (
        100 * comparison[f"adapt_aceptadas_{run}"] / comparison[f"analizadas_{run}"].replace(0, np.nan)
    )

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, metric, title in [
    (axes[0, 0], "tasa_phot_validas", "PHOT con magnitud y error definidos"),
    (axes[0, 1], "tasa_merr", "Pasan el corte MERR ≤ 0.20"),
    (axes[1, 0], "tasa_cielo_limpio", "Anillos fijos clasificados como limpios"),
    (axes[1, 1], "tasa_adapt_aceptada", "Aceptadas con anillo adaptativo"),
]:
    old_col, new_col = f"{metric}_anterior", f"{metric}_nuevo"
    # La función usa las columnas estándar; las tasas se grafican en el mismo formato de pares.
    x = np.arange(len(comparison))
    old_values = comparison[old_col].to_numpy(dtype=float)
    new_values = comparison[new_col].to_numpy(dtype=float)
    for i, (old_value, new_value) in enumerate(zip(old_values, new_values)):
        ax.plot([i - 0.12, i + 0.12], [old_value, new_value], color="#B8B8B8", lw=1, zorder=1)
    ax.scatter(x - 0.12, old_values, color="#3B6FB6", marker="o", s=34, label="Anterior", zorder=2)
    ax.scatter(x + 0.12, new_values, color="#E07A27", marker="s", s=34, label="Nueva", zorder=2)
    ax.set_xticks(x, comparison["chip"].astype(str))
    ax.set_xlabel("Chip")
    ax.set_ylabel("porcentaje (%)")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)
fig.suptitle("Rendimiento relativo de detección y selección", y=1.02, fontsize=14)
save_show(fig, "07_tasas_de_seleccion.png")

summary_change = pd.DataFrame({
    "métrica": ["Δ magnitud mediana (mag)", "|Δ magnitud| mediana (mag)", "MERR adaptativo mediano (mag)", "Δ contaminación mediana (puntos porcentuales)"],
    "mediana_anterior_chips_2_8": [
        comparison["delta_mag_anterior"].median(),
        comparison["abs_delta_mag_anterior"].median(),
        comparison["merr_adapt_anterior"].median(),
        100 * comparison["delta_fout_anterior"].median(),
    ],
    "mediana_nueva_chips_2_8": [
        comparison["delta_mag_nuevo"].median(),
        comparison["abs_delta_mag_nuevo"].median(),
        comparison["merr_adapt_nuevo"].median(),
        100 * comparison["delta_fout_nuevo"].median(),
    ],
})
summary_change["nueva_menos_anterior"] = (
    summary_change["mediana_nueva_chips_2_8"] - summary_change["mediana_anterior_chips_2_8"]
)
display(summary_change.round(4))

print(f"Tabla exportada: {NEW_ROOT / 'comparacion_corridas_chip2_8.csv'}")
print(f"Figuras guardadas en: {FIG_DIR}")

![Tasas relativas de selección](dataSextansA_multiplicadores_FWHM/figuras_comparacion/07_tasas_de_seleccion.png)

## 9. Alcance e interpretación

- Las tablas y figuras comparan solo chips 2–8. No se extrapolan resultados al chip 1.
- La corrida anterior del chip 2 tiene un modo y una muestra de estrellas/cielo distintos a la nueva; las diferencias de ese chip incluyen ambos cambios.
- Para los chips 3–8, la modificación solicitada fue el radio de apertura. El chip 6 también mueve el radio interno del cielo a 7 × FWHM para que el anillo quede fuera de la apertura; revisa esa diferencia al interpretar el resultado.
- Las magnitudes y errores adaptativo/fijo se resumen sobre las fuentes cuyo anillo de cielo fue desplazado y cuya fotometría es válida. El tamaño de esa muestra se muestra por separado.
- Las gráficas de `f_out` usan porcentajes; el cambio `Δ f_out` usa puntos porcentuales.

Los archivos detallados por chip siguen disponibles en `dataSextansA_multiplicadores_FWHM/chips/chipN`; el cuaderno no modifica los resultados de ninguna corrida.